In [2]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print("Webcam error.")
else:
    print("Ready!")

Ready!


In [4]:
print("Press 'q' in the camera window to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    frame = cv2.flip(frame, 1)
    
    cv2.rectangle(frame, (350, 100), (600, 350), (255, 0, 0), 2)
    roi = frame[100:350, 350:600]
    
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    
    lower_skin = np.array([0, 20, 70], dtype=np.uint8)
    upper_skin = np.array([20, 255, 255], dtype=np.uint8)
    
    mask = cv2.inRange(hsv, lower_skin, upper_skin)
    mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=2)
    mask = cv2.GaussianBlur(mask, (5, 5), 100)
    
    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    
    reply = "Put hand in blue box..."
    
    if contours:
        max_contour = max(contours, key=cv2.contourArea)
        
        if cv2.contourArea(max_contour) > 2000:
            hull = cv2.convexHull(max_contour, returnPoints=False)
            defects = cv2.convexityDefects(max_contour, hull)
            
            finger_count = 0
            
            if defects is not None:
                for i in range(defects.shape[0]):
                    s, e, f, d = defects[i, 0]
                    start = tuple(max_contour[s][0])
                    end = tuple(max_contour[e][0])
                    far = tuple(max_contour[f][0])
                    
                    a = np.linalg.norm(np.array(end) - np.array(start))
                    b = np.linalg.norm(np.array(far) - np.array(start))
                    c = np.linalg.norm(np.array(end) - np.array(far))
                    
                    angle = np.arccos((b**2 + c**2 - a**2) / (2 * b * c)) * 57.29
                    
                    if angle <= 90 and d > 10000:
                        finger_count += 1
                        cv2.circle(roi, far, 5, [0, 0, 255], -1)
                
                if finger_count > 0:
                    finger_count += 1
                    
            if finger_count == 5:
                reply = "Greetings! "
            elif finger_count == 2:
                reply = "Peace! "
            elif finger_count == 1:
                reply = "Thumbs up! "
            elif finger_count == 0:
                
                if cv2.contourArea(max_contour) < 6000:
                    reply = "What do you want? "
                else:
                    reply = "Fist! "

    cv2.putText(frame, reply, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    cv2.imshow("Pure OpenCV Tracker", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

Press 'q' in the camera window to quit.


In [5]:
cap.release()
print("Webcam turned off.")

Webcam turned off.
